# Anomaly Detection — ERFNet (MSP · MaxLogit · MaxEntropy)


## Cell 1 — Mount Drive & verify GPU

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/AnomalyProject/results', exist_ok=True)

gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print('Drive mounted.')
print('GPU:', gpu if gpu else '❌ NOT FOUND — change runtime to T4 GPU!')

Mounted at /content/drive
Drive mounted.
GPU: Tesla T4


## Cell 2 — Install packages & clone repo

In [2]:
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'gdown', 'Pillow', 'numpy',
                'torch', 'torchvision'], check=True)
print('✓ Packages installed')

REPO = '/content/MaskArchitectureAnomaly_CourseProject'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone',
        'https://github.com/AlessandroMarinai/MaskArchitectureAnomaly_CourseProject.git'],
        check=True)
print('✓ Repo ready')
print('eval/ contents:', os.listdir(os.path.join(REPO, 'eval')))

✓ Packages installed
✓ Repo ready
eval/ contents: ['eval_forwardTime.py', 'results.txt', 'iouEval.py', 'dataset.py', '__pycache__', 'eval_cityscapes_server.py', 'eval_iou.py', 'evalAnomaly.py', 'erfnet.py', 'README.md', 'transform.py', 'eval_cityscapes_color.py', 'erfnet_nobn.py']


## Cell 3 — Download & extract datasets
Downloads `Anomaly_Validation_Datasets.zip` from the course Drive folder.

In [3]:
import subprocess, sys, os, zipfile

DATASET_DIR = '/content/datasets'
ZIP_PATH    = '/content/Anomaly_Validation_Datasets.zip'
FOLDER_ID   = '1q2vHUzora2nP52fP50zmoQAykWuwoGav'

os.makedirs(DATASET_DIR, exist_ok=True)

# ── Step 1: list folder to find zip file ID ────────────────────────────────
if not os.path.isfile(ZIP_PATH):
    print('Listing Drive folder to find zip ID...')
    r = subprocess.run(
        ['gdown', '--folder', FOLDER_ID, '--list', '--remaining-ok'],
        capture_output=True, text=True)
    print(r.stdout)

    # Parse the zip file ID from gdown output
    zip_id = None
    for line in r.stdout.splitlines():
        if 'Anomaly_Validation_Datasets' in line and 'id:' in line:
            zip_id = line.split('id:')[-1].strip()
            break

    if zip_id is None:
        # Fallback: try downloading entire folder (zip may be at top level)
        print('Could not auto-detect zip ID — downloading full folder...')
        subprocess.run(['gdown', '--folder', FOLDER_ID,
                        '-O', '/content/drive_dl', '--remaining-ok'], check=True)
        for root, _, files in os.walk('/content/drive_dl'):
            for f in files:
                if f.endswith('.zip'):
                    import shutil
                    shutil.move(os.path.join(root, f), ZIP_PATH)
                    print(f'Found zip: {f}')
                    break
    else:
        print(f'Zip ID found: {zip_id}')
        subprocess.run(['gdown', zip_id, '-O', ZIP_PATH], check=True)

print('✓ Zip present at', ZIP_PATH)

# ── Step 2: extract ────────────────────────────────────────────────────────
expected = os.path.join(DATASET_DIR, 'RoadAnomaly21')
if not os.path.isdir(expected):
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATASET_DIR)
    print('✓ Extracted')
else:
    print('✓ Already extracted')

# ── Step 3: verify tree ────────────────────────────────────────────────────
print('\nDataset tree:')
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    if level > 2: continue
    print('  ' * level + os.path.basename(root) + '/')
    if level == 2:
        exts = set(os.path.splitext(f)[1] for f in files)
        print('  ' * (level+1) + f'{len(files)} files  extensions: {exts}')

Listing Drive folder to find zip ID...

Could not auto-detect zip ID — downloading full folder...
Found zip: Anomaly_Validation_Datasets.zip
✓ Zip present at /content/Anomaly_Validation_Datasets.zip
Extracting...
✓ Extracted

Dataset tree:
datasets/
  __MACOSX/
    Validation_Dataset/
      1 files  extensions: {'.DS_Store'}
  Validation_Dataset/
    RoadObsticle21/
      1 files  extensions: {''}
    FS_LostFound_full/
      1 files  extensions: {''}
    RoadAnomaly21/
      1 files  extensions: {''}
    RoadAnomaly/
      1 files  extensions: {''}
    fs_static/
      1 files  extensions: {''}


## Cell 4 — Write `eval/evalAnomaly.py`
`%%writefile` saves this cell to disk — **never comment it out**.

In [4]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly.py
# evalAnomaly.py — pixel-wise anomaly scoring with ERFNet
# Methods: MSP, MaxLogit, MaxEntropy
import os, sys, glob, argparse
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from erfnet import ERFNet

NUM_CLASSES   = 20
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def load_model(path, device):
    """Load ERFNet weights. encoder.output_conv is training-only; skip it."""
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Weights not found: {path}')
    model = ERFNet(NUM_CLASSES)
    state = torch.load(path, map_location=device)
    if 'state_dict' in state:
        state = state['state_dict']
    state = {k.replace('module.', ''): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    expected_missing = {'encoder.output_conv.weight', 'encoder.output_conv.bias'}
    real_missing = set(missing) - expected_missing
    if real_missing:
        raise RuntimeError(f'Unexpected missing keys: {real_missing}')
    if unexpected:
        print(f'[WARNING] Unexpected keys in checkpoint: {unexpected}')
    return model.to(device).eval()


def preprocess(path):
    """Load RGB image → normalised (1,3,H,W) tensor."""
    img = Image.open(path).convert('RGB')
    t = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return t(img).unsqueeze(0)


@torch.no_grad()
def get_logits(model, tensor, device):
    """Forward pass → raw logits (C,H,W)."""
    return model(tensor.to(device)).squeeze(0)


# ── Scoring methods (higher value = more anomalous) ────────────────────────

def score_msp(logits):
    """
    Maximum Softmax Probability (MSP) — Hendrycks & Gimpel, 2017.
    score = 1 - max_c softmax(logits)_c
    Low max-probability pixels are uncertain → likely anomalous.
    """
    return (1.0 - F.softmax(logits, dim=0).max(dim=0).values).cpu().numpy()


def score_maxlogit(logits):
    """
    MaxLogit — Hendrycks et al., 2022.
    score = -max_c logits_c
    Low max-logit pixels have no strong class signal → likely anomalous.
    """
    return (-logits.max(dim=0).values).cpu().numpy()


def score_maxentropy(logits):
    """
    Softmax Entropy — Chan et al. (Meta-OOD), 2021.
    score = -sum_c p_c * log(p_c)
    High entropy → uniform distribution → network is confused → anomaly.
    Reference implementation:
      https://github.com/SegmentMeIfYouCan/road-anomaly-benchmark/blob/master/methods/baselines.py#L85
    """
    p = F.softmax(logits, dim=0)
    return (-(p * torch.log(p.clamp(min=1e-10))).sum(dim=0)).cpu().numpy()


METHODS = {
    'msp':        score_msp,
    'maxlogit':   score_maxlogit,
    'maxentropy': score_maxentropy,
}


def find_images(pattern):
    """Glob for images; try .png then .jpg if nothing found."""
    paths = sorted(glob.glob(pattern))
    if not paths:
        alt = pattern.replace('*.png', '*.jpg')
        paths = sorted(glob.glob(alt))
    if not paths:
        alt = pattern.replace('*.png', '*.*')
        paths = sorted(glob.glob(alt))
    return paths


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--input',   required=True, help='Glob pattern for images')
    ap.add_argument('--method',  default='msp', choices=METHODS)
    ap.add_argument('--weights', default='../trained_models/erfnet_pretrained.pth')
    ap.add_argument('--output',  default='./scores')
    args = ap.parse_args()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'[INFO] Device={device}  Method={args.method}')

    model = load_model(args.weights, device)
    paths = find_images(args.input)
    if not paths:
        print(f'[ERROR] No images found: {args.input}'); return
    print(f'[INFO] {len(paths)} image(s) found.')

    os.makedirs(args.output, exist_ok=True)
    fn = METHODS[args.method]

    for i, p in enumerate(paths):
        stem  = os.path.splitext(os.path.basename(p))[0]
        score = fn(get_logits(model, preprocess(p), device))
        np.save(os.path.join(args.output, f'{stem}_{args.method}.npy'), score)
        if (i + 1) % 10 == 0 or (i + 1) == len(paths):
            print(f'  [{i+1}/{len(paths)}]')

    print(f'[DONE] Saved to {args.output}')


if __name__ == '__main__':
    main()

Overwriting /content/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly.py


## Cell 5 — Write `eval/metrics.py`

In [5]:
%%writefile /content/MaskArchitectureAnomaly_CourseProject/eval/metrics.py
# metrics.py — AuPRC and FPR@95 for anomaly segmentation evaluation
import os, glob
import numpy as np
from PIL import Image
from sklearn.metrics import average_precision_score, roc_curve


def load_pairs(scores_dir, masks_dir, method, mask_ext=None):
    """
    Load matching score maps (.npy) and ground-truth masks.

    Mask convention (SMIYC / FS datasets):
      255  → void / ignore
      > 0  → anomaly  (positive)
      0    → normal   (negative)
    """
    npy_files = sorted(glob.glob(os.path.join(scores_dir, f'*_{method}.npy')))
    if not npy_files:
        raise FileNotFoundError(
            f'No *_{method}.npy files in {scores_dir}')

    # Auto-detect mask extension if not given
    if mask_ext is None:
        for ext in ('png', 'jpg', 'jpeg', 'webp'):
            sample = os.path.join(
                masks_dir,
                os.path.basename(npy_files[0]).replace(f'_{method}.npy', f'.{ext}'))
            if os.path.isfile(sample):
                mask_ext = ext
                break
        if mask_ext is None:
            mask_ext = 'png'

    S, L = [], []
    for npy_path in npy_files:
        stem      = os.path.basename(npy_path).replace(f'_{method}.npy', '')
        mask_path = os.path.join(masks_dir, f'{stem}.{mask_ext}')

        if not os.path.isfile(mask_path):
            print(f'  [SKIP] mask not found: {mask_path}')
            continue

        score = np.load(npy_path).astype(np.float32)
        mask  = np.array(Image.open(mask_path))

        # Resize score to mask resolution if they differ
        if score.shape != mask.shape:
            score = np.array(
                Image.fromarray(score).resize(
                    (mask.shape[1], mask.shape[0]), Image.BILINEAR))

        valid = mask != 255
        if valid.sum() == 0:
            print(f'  [SKIP] all void: {mask_path}')
            continue

        S.append(score[valid].ravel())
        L.append((mask[valid] > 0).ravel().astype(np.int32))

    if not S:
        raise RuntimeError(f'No valid (score, mask) pairs loaded from {scores_dir}')

    return np.concatenate(S), np.concatenate(L)


def compute_auprc(scores, labels):
    """Area under Precision-Recall Curve (0–1)."""
    return float(average_precision_score(labels, scores))


def fpr95(scores, labels):
    """False Positive Rate at 95 % True Positive Rate (0–1)."""
    fpr_arr, tpr_arr, _ = roc_curve(labels, scores)
    idx = np.searchsorted(tpr_arr, 0.95)
    return float(fpr_arr[min(idx, len(fpr_arr) - 1)])

Writing /content/MaskArchitectureAnomaly_CourseProject/eval/metrics.py


## Cell 6 — Configuration
Paths match the **exact** folder names from the downloaded zip.

In [8]:
import os

REPO_ROOT    = '/content/MaskArchitectureAnomaly_CourseProject'
WEIGHTS      = f'{REPO_ROOT}/trained_models/erfnet_pretrained.pth'
EVAL_SCRIPT  = f'{REPO_ROOT}/eval/evalAnomaly.py'
RESULTS_ROOT = '/content/drive/MyDrive/AnomalyProject/results'
DATASET_BASE = '/content/datasets/Validation_Dataset'

# ── Exact folder names confirmed from the zip ─────────────────────────────
# Each entry: dataset_key → (image_glob, masks_dir)
DATASETS = {
    'RoadAnomaly21': (
        f'{DATASET_BASE}/RoadAnomaly21/images/*.png',
        f'{DATASET_BASE}/RoadAnomaly21/labels_masks'),
    'RoadObsticle21': (
        f'{DATASET_BASE}/RoadObsticle21/images/*.webp',
        f'{DATASET_BASE}/RoadObsticle21/labels_masks'),
    'FS_LostFound': (
        f'{DATASET_BASE}/FS_LostFound_full/images/*.png',
        f'{DATASET_BASE}/FS_LostFound_full/labels_masks'),
    'fs_static': (
        f'{DATASET_BASE}/fs_static/images/*.jpg',
        f'{DATASET_BASE}/fs_static/labels_masks'),
    'RoadAnomaly': (
        f'{DATASET_BASE}/RoadAnomaly/images/*.jpg',
        f'{DATASET_BASE}/RoadAnomaly/labels_masks'),
}

METHODS = ['msp', 'maxlogit', 'maxentropy']

# ── Sanity check: verify all image folders exist ──────────────────────────
print('Dataset verification:')
import glob as _glob
for ds, (img_glob, mask_dir) in DATASETS.items():
    imgs  = _glob.glob(img_glob)
    # also try jpg if no png found
    if not imgs:
        imgs = _glob.glob(img_glob.replace('*.png','*.jpg', '*.webp'))
    masks = os.path.isdir(mask_dir)
    status = '✓' if imgs and masks else '✗'
    print(f'  {status} {ds:20s}  images={len(imgs):4d}  masks_dir={masks}')
    if imgs:
        # Update glob to correct extension for later use
        ext = os.path.splitext(imgs[0])[1]
        DATASETS[ds] = (img_glob.replace('*.png', f'*{ext}'), mask_dir)

Dataset verification:
  ✓ RoadAnomaly21         images=  10  masks_dir=True
  ✓ RoadObsticle21        images=  30  masks_dir=True
  ✓ FS_LostFound          images= 100  masks_dir=True
  ✓ fs_static             images=  30  masks_dir=True
  ✓ RoadAnomaly           images=  60  masks_dir=True


## Cell 7 — Run scoring (saves .npy maps to Drive)

In [11]:
import subprocess, sys, os

os.makedirs(RESULTS_ROOT, exist_ok=True)

for method in METHODS:
    for ds_name, (img_glob, _) in DATASETS.items():
        out_dir = os.path.join(RESULTS_ROOT, f'{method}_{ds_name}')
        os.makedirs(out_dir, exist_ok=True)
        print(f'\n► {method:12s}  {ds_name}')

        r = subprocess.run(
            [sys.executable, EVAL_SCRIPT,
             '--input',   img_glob,
             '--method',  method,
             '--weights', WEIGHTS,
             '--output',  out_dir],
            capture_output=True, text=True
        )
        out = (r.stdout + r.stderr).strip()
        print(out[-500:] if len(out) > 500 else out)
        if r.returncode != 0:
            print(f'  ⚠ Exit code {r.returncode}')

print('\n✅ Scoring complete.')


► msp           RoadAnomaly21
[INFO] Device=cuda  Method=msp
[INFO] 10 image(s) found.
  [10/10]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_RoadAnomaly21

► msp           RoadObsticle21
[INFO] Device=cuda  Method=msp
[INFO] 30 image(s) found.
  [10/30]
  [20/30]
  [30/30]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_RoadObsticle21

► msp           FS_LostFound
[INFO] Device=cuda  Method=msp
[INFO] 100 image(s) found.
  [10/100]
  [20/100]
  [30/100]
  [40/100]
  [50/100]
  [60/100]
  [70/100]
  [80/100]
  [90/100]
  [100/100]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_FS_LostFound

► msp           fs_static
[INFO] Device=cuda  Method=msp
[INFO] 30 image(s) found.
  [10/30]
  [20/30]
  [30/30]
[DONE] Saved to /content/drive/MyDrive/AnomalyProject/results/msp_fs_static

► msp           RoadAnomaly
[INFO] Device=cuda  Method=msp
[INFO] 60 image(s) found.
  [10/60]
  [20/60]
  [30/60]
  [40/60]
  [50/60]
  [60/60]
[DONE] S

## Cell 8 — Compute metrics & print results table

In [12]:
import sys
sys.path.insert(0, f'{REPO_ROOT}/eval')
from metrics import load_pairs, compute_auprc, fpr95

W = (14, 18, 9, 9)
print(f"{'Method':<{W[0]}} {'Dataset':<{W[1]}} {'AuPRC':>{W[2]}} {'FPR@95':>{W[3]}}")
print('-' * sum(W))

for method in METHODS:
    for ds_name, (_, masks_dir) in DATASETS.items():
        scores_dir = os.path.join(RESULTS_ROOT, f'{method}_{ds_name}')
        try:
            S, L  = load_pairs(scores_dir, masks_dir, method)
            auprc = compute_auprc(S, L) * 100
            f95   = fpr95(S, L) * 100
            print(f'{method:<{W[0]}} {ds_name:<{W[1]}} {auprc:>{W[2]-1}.1f}% {f95:>{W[3]-1}.1f}%')
        except Exception as e:
            print(f'{method:<{W[0]}} {ds_name:<{W[1]}}  ERROR: {e}')

Method         Dataset                AuPRC    FPR@95
--------------------------------------------------
msp            RoadAnomaly21          15.8%     89.4%
msp            RoadObsticle21          0.6%     96.4%
msp            FS_LostFound            0.3%     91.5%
msp            fs_static               1.1%     98.1%
msp            RoadAnomaly            10.8%     89.3%
maxlogit       RoadAnomaly21          15.4%     87.7%
maxlogit       RoadObsticle21          2.5%     89.8%
maxlogit       FS_LostFound            0.3%     86.1%
maxlogit       fs_static               1.2%     91.9%
maxlogit       RoadAnomaly            10.9%     87.3%
maxentropy     RoadAnomaly21          15.2%     89.5%
maxentropy     RoadObsticle21          0.7%     96.5%
maxentropy     FS_LostFound            0.3%     91.5%
maxentropy     fs_static               1.1%     98.1%
maxentropy     RoadAnomaly            10.4%     89.5%
